In [0]:
from pyspark.sql import functions as F

CATALOG = "supplychain"
LANDING = f"/Volumes/{CATALOG}/landing/erp"
CHECKPOINTS = f"/Volumes/{CATALOG}/landing/checkpoints"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.bronze")

# Chaque élément de data reste un JSON brut (string) : rien n'est transformé en Bronze
PAGE_SCHEMA = "data ARRAY<STRING>, _run_id STRING, _fetched_at STRING"


def autoload_entity(entity: str, table_name: str) -> None:
    table = f"{CATALOG}.bronze.{table_name}"

    pages = (spark.readStream
             .format("cloudFiles")
             .option("cloudFiles.format", "text")
             .option("wholetext", True)                 # un fichier = une page entière
             .option("pathGlobFilter", "page_*.json")   # seulement les pages
             .option("recursiveFileLookup", True)       # parcourt ingest_date=…/run_id=…
             .load(f"{LANDING}/{entity}"))

    records = (pages
               .select(F.from_json("value", PAGE_SCHEMA).alias("p"),
                       F.col("_metadata.file_path").alias("_file"))
               .select("p._run_id", "p._fetched_at", "_file", F.explode("p.data").alias("payload"))
               .select(F.get_json_object("payload", "$.id").alias("source_id"),
                       F.get_json_object("payload", "$.audit.updated_at").alias("source_updated_at"),
                       "payload", "_run_id", "_fetched_at", "_file",
                       F.current_timestamp().alias("_ingest_ts")))

    (records.writeStream
        .option("checkpointLocation", f"{CHECKPOINTS}/{table_name}")
        .trigger(availableNow=True)                     # traite le nouveau puis s'arrête
        .toTable(table)
        .awaitTermination())

    print(f"{table} : {spark.table(table).count()} lignes au total")


autoload_entity("suppliers", "erp_suppliers")
autoload_entity("parts", "erp_parts")
autoload_entity("purchase-orders", "erp_purchase_orders")